# **CS TA Application - Winter 2026**

Hello MEET Comp-Sci Team, I hope you are all doing well!
This is my task submission for the 2026 CS TA applications.

This project consists of several features covering external APIs, text flair, a simple algorithm which attempts to widen the input possibilities of a non-AI chatbot, and a few other miscellaneous features. So without further ado, here is the explanation of the code and the features, and the future steps!

Below I will provide a general explanation of each section and the important systems in the bot in short; longer and more in-depth explanations covering each function and their systems will also be provided afterwards.

## ***Overview of Key Features***

### *Printing & Text Output*
For this project, one of the main goals I wanted to work on was having a dynamic printing system for the CLI (Command-Line Interface). This meant having animated printing and a skipping system for the printing.
The final implementation consists of a few helper functions and a main threader which connects both dynamic printing and animation skipping together.

*Note: Printing speed is able to be changed through the settings option of the chatbot.*

### *API Feature - APOD (Astronomy Picture of the Day)*
This feature comes from a personal interest in space and NASA's open-source libraries (and my previous usage of this API in my Year 2 Summer Task!). This API delivers pictures relating to astronomy, providing context, a URL for the image, and a Title. You are also able to request images from certain dates, but I did not utilize this feature (yet!).

In the chatbot, I allowed the user to request the current picture of the day after inputting their own API key. The output is provided in the format of **Title**, **Explanation**, and **URL**.

*Note: The user is also able to change their API key through the settings option.*

### *Response System*
To make the chatbot responses more dynamic without using AI, I implemented my own algorithm. Using the input sentence, it attempts to determine what is the most optimal response from the database given to the chatbot (which can be customized and expanded upon by developers!). When an option is chosen, the bot picks a random reply from the list of provided responses and displays it to the user.

*(For a more in-depth explanation of this system, refer to the longer explanation section below!)*

### *Settings Menu*
To improve user experience, I integrated a simple settings menu which gives the user the ability to change three settings:
1. Their **username** (what the chatbot refers to them as)
2. Their **API key** for the APOD feature
3. The **speed** of the text animation

## *Future Improvements & Thoughts*

While I am truly proud of the current state of the chatbot, I believe many improvements can be made to every system provided here, as they can be expanded upon:

1. **APOD Integration:** This system can improve by integrating a new option for viewing previous APODs by requesting a specific date from the API, or displaying the image either directly or by using [ASCII Art](https://www.asciiart.eu/faq).

2. **Response System Expansion:** The largest improvement and expansion I would love to pursue is the expansion of the response system. As the rise of AI reaches all developers, the need for a simpler system which uses premade responses is a great breath of fresh air. But the current existing system is quite simple and considers a basic algorithm which fails to account for main ideas in sentences and text, only using key-words as its metric of a response gauge. Improving and creating a system that is able to adapt to all forms of text through an analytical understanding of language structure and deliver a response from its own system would be a great improvement and a truly interesting project to undertake!

3. **Chat History**: One of the most important features in any application and assistive tool is the ability to save previous chats and actions done; this helps organise the user, retrieve already asked information and helps users greatly.

4. **Quality of Life:** Many QoL changes to the current system—such as expanded settings, additional preferences, and an auto-correct system—can be integrated to improve the general user experience.

To view the full code of the project please open the attached python file! It currently contains the response data and keys for simplicity.

## ***Detailed Explanation: Printing & Text Output***

For this project, one of the main goals I wanted to work on was having a dynamic printing system for the CLI (Command-Line Interface). This meant having animated printing and a skipping system for the printing.
The final implementation consists of a few helper functions and a main threader which connects both dynamic printing and animation skipping together.

First come the helper functions:

In [ ]:
import sys
import time
import platform
import os

# Listener function to check for enter key press, skipping text animation
def t_onpress_print(key): 
    global skipCheck
    if key == keyboard.Key.enter: 
        skipCheck[0] = True
        # print('\nLISTENER STOPPED\n')
        return False  # Stop the listener

# Print function that prints character by character with a delay (FLAIR)
def printtime(in_str, speed=1): 
    global skipCheck
    for i in range(len(in_str)):
        if skipCheck[0]:
            # print('\SKIPPED\n')
            sys.stdout.write(in_str[i:])
            sys.stdout.flush()
            return
        sys.stdout.write(in_str[i])
        sys.stdout.flush()
        time.sleep(speed)

# Clears the text using single line removal (currently backup to clear() function)
def cleartxt(): 
    time.sleep(1)
    sys.stdout.write("\r" + " " * 100) 
    sys.stdout.write("\r")
    sys.stdout.flush()

# Reflects the clear function to the appropriate OS
match platform.system():
    case "Windows":
        clear = lambda: os.system('cls') #Clears the console
    case "Linux" | "Darwin":
        clear = lambda: os.system('clear') #Clears the console
    case _:
        clear = lambda: cleartxt()


The three helper functions seen above control the internal needs of the output/printing system. They consist of:

- **The Press Event:** Handles the skip event by checking if the correct key is pressed, stopping the listener.
- **The Screen Clear Function/s:** Handles clearing and cleaning the screen/output by picking the correct command after detecting the operating system (`Windows`, `Linux`, or `MacOS/Darwin`). It provides a cleaner view and can be triggered manually through the chatbot.
- **The Dynamic Printing Function:** The core of the dynamic printing system. It prints text using a typing animation by writing each character with an inputted speed via terminal stdout. It also ensures that when an animation is skipped, all animations halt and the remaining text displays immediately.

These all work in parallel with the threader to execute dynamic printing.

In [ ]:
import threading
from pynput import keyboard
import msvcrt

def flush_input():
    while msvcrt.kbhit():
        msvcrt.getch()
        
textSpeed=0.045
# Threader function allowing both the text animation and the listener to run simultaneously, allowing for skipping of text animation
def print_threader(in_str, speed=1):
    speed = textSpeed * speed
    global skipCheck
    skipCheck = [False]
    # print(speed*len(in_str)-0.5)
    t_print = threading.Thread(target=printtime, args=(in_str, speed))
    t_print.start()
    with keyboard.Listener(on_press=t_onpress_print,) as listener:    
        listener.join(timeout=(speed*len(in_str)-0.5))
    t_print.join()

    flush_input()
    return True

### *The Text Threader*

Shown above is the text threader function. This function creates two threads using both a standard thread and a Listener from `pynput`.

When invoked, it starts both threads, prints the input using the helper functions above, and manages all threads to resume synchronous execution smoothly.

#### Run Order Overview

```text
                                           /-> Thread 1 Starts -> Prints Text & Awaits Update -> Thread Joins \
                                          /                                                                   \
Start -> Variables Created ---------------<                                                                    -----> Cleanup + End
                                          \                                                                   /
                                           \-> Listener Starts -> Awaits Input & Updates Thread 1 -> Joins ---/
```

## ***Detailed Explanation: Response & Decision Process***

The chatbot's response system consists of five main functions:
1. **Sentence Setup**: This function takes the input sentence given by the user and removes all unnecesary sections (punctuation, grammar, etc.), leaving a list of barebone key-words used for scoring.
2. **Scoring**: This function compares the list of words (the sentence) and compares them with the lists provided for replies (keys), calculating the raw score, then providing a percentage calculated as such: ((raw score) / ((sum of weights)*0.72))*100. This simple formula provides a relatively accurate system predicting the response required for the input!
3. **Output Selection**: This function compares and finds the (reply) key with the highest score and returns it as the selected output
4. **Threshhold Check**: Compares the selected output with a threshhold minimum to ensure that the reply is accurate.

These functions are done in succession and are built to handle inputs which result from the previous function's input, leading to a very smooth execution. 

In [ ]:
from math import ceil,floor
nonkey_wordlist = [] # Empty for Notebook, it is shown in the full code above!


# Rounds the input to nearest integer
def round_num(num):
    return ceil(num) if num-floor(num) >= 0.5 else floor(num)

"""CODE EXPLAINED ABOVE BEGINS HERE"""

# Removes unnecessary punctuation and non-key words from the input sentence, returning a list of key words for algorithm processing
def strip_sentence(sentence, wordlist=nonkey_wordlist):
    # Remove punctuation and convert to lowercase
    sentence = sentence.lower()
    for char in ['.', ',', '!', '?', ';', ':', '"', "'", '(', ')', '[', ']', '{', '}', '-', '_']:
        sentence = sentence.replace(char, '')
    sentence = sentence.strip().split(' ')
    for word in sentence:
        word = word.strip()
        if word in wordlist:
            sentence.remove(word)
    return sentence

# Analyzes the input with the keys, returning a list of scores for each key with a non-zero score 
# Explained in the README file
def get_similarity_scores(user_input, keys):
    scores = []
    score_index=0
    for key in keys:
        w_sum=0
        scores.append([key, 0])
        for word in user_input:
            found_inKey=False
            for key_word, weight in keys[key]:
                if word == key_word: scores[score_index][1] += weight ; found_inKey=True
            if not found_inKey: w_sum += 1 
            else: found_inKey=False
        if scores[score_index][1] == 0:
            scores.remove(scores[score_index])
        else:
            for key_word, weight in keys[key]: w_sum+=weight if weight>1 else 0
            scores[score_index][1] = round_num((scores[score_index][1] / (w_sum*0.72))*100) if w_sum > 0 else scores[score_index][1]
            score_index += 1
    return scores


# Returns the maximum score from the list of scores, or None if the list is empty
def get_best_output(scores):
    return max(scores, key=lambda x: x[1]) if scores else None

# Checks if the best score meets or exceeds the threshold to be considered a valid match
def check_threshold(best_score, threshold=15):
    return best_score[1] >= threshold if best_score else False

Thank you for reading!
If you have any comments or questions don't hesitate to email me or send me a message on linkedin!

### *[E-Mail](mailto:joodnasserbusiness@gmail.com)*

### *[LinkedIn](https://www.linkedin.com/in/joud-nasser-b25042249)*